In [ ]:
!pip install diffusers

In [ ]:
from diffusers import UNet2DModel, DDIMScheduler
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

class DirectionRegressor(nn.Module):
    """
    Reconstructor network that predicts which latent direction was applied
    to the bottleneck representation.
    """
    def __init__(self, bottleneck_dim, num_directions):
        super().__init__()
        # Calculate flattened input size from bottleneck dimensions
        if isinstance(bottleneck_dim, tuple):
            # If bottleneck_dim is a tuple (C, H, W), flatten it
            flat_size = bottleneck_dim[0] * bottleneck_dim[1] * bottleneck_dim[2]
        else:
            # If already a single number
            flat_size = bottleneck_dim

        # Architecture: flatten bottleneck -> FC layers -> direction classification
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, num_directions)
        )

    def forward(self, x):
        return self.model(x)

class AuxiliaryNetwork(nn.Module):
    """
    Auxiliary network that generates interpretable h-space directions
    through factorization.
    """
    def __init__(self, bottleneck_channels, num_directions, bottleneck_spatial_dim=None):
        super().__init__()
        self.num_directions = num_directions
        self.bottleneck_channels = bottleneck_channels

        if bottleneck_spatial_dim is None:
            # Default spatial dimensions if not specified
            bottleneck_spatial_dim = (8, 8)  # Common for 256x256 images
        else:
            bottleneck_spatial_dim = tuple(bottleneck_spatial_dim)

        self.spatial_dim = bottleneck_spatial_dim

        # A matrix maps one-hot encoded directions to channel-wise shifts
        # Initializing with small random values
        self.A = nn.Parameter(
            torch.randn(num_directions, bottleneck_channels) * 0.02
        )

        # Spatial attention masks - one per direction to allow for spatially-varied edit
        self.spatial_masks = nn.Parameter(
            torch.ones(num_directions, 1, bottleneck_spatial_dim[0], bottleneck_spatial_dim[1])
        )

        # Optional: add scale parameters to control strength of edits
        self.scales = nn.Parameter(torch.ones(num_directions))

    def forward(self, bottleneck, direction_idx=None, magnitude=1.0, random_directions=False, binary_vectors=None):
        """
        Apply a transformation to the bottleneck features.

        Args:
            bottleneck: The h-space features from UNet's bottleneck (B, C, H, W)
            direction_idx: Index of the specific direction to apply (int or None)
            magnitude: Strength of the edit (float)
            random_directions: Whether to apply random directions (boolean)
            binary_vectors: Pre-generated binary vectors for directions (tensor or None)

        Returns:
            Modified bottleneck features
        """
        batch_size = bottleneck.shape[0]
        device = bottleneck.device

        # Choose how to generate direction vectors
        if binary_vectors is not None:
            # Use pre-generated binary vectors
            direction_vectors = binary_vectors
        elif random_directions:
            # Generate random binary vectors (0 or 1 entries)
            direction_vectors = torch.zeros(batch_size, self.num_directions, device=device)
            for i in range(batch_size):
                # Randomly select 1-3 directions per sample
                num_active = torch.randint(1, min(4, self.num_directions + 1), (1,)).item()
                active_indices = torch.randperm(self.num_directions)[:num_active]
                direction_vectors[i, active_indices] = 1.0
        elif direction_idx is not None:
            # Use a specific direction for all samples in the batch
            direction_vectors = torch.zeros(batch_size, self.num_directions, device=device)
            direction_vectors[:, direction_idx] = 1.0
        else:
            # Default: identity transformation (no change)
            return bottleneck

        # Apply A matrix to get channel-wise shifts: (B, D) @ (D, C) -> (B, C)
        channel_shifts = direction_vectors @ (self.A * self.scales.unsqueeze(1))

        # Reshape for broadcasting: (B, C, 1, 1)
        channel_shifts = channel_shifts.unsqueeze(-1).unsqueeze(-1)

        # Create delta_h for each sample in batch
        delta_h = torch.zeros_like(bottleneck)

        # Instead of batched matrix multiplication which was causing dimension issues,
        # apply masks individually for each sample in the batch
        for i in range(batch_size):
            # Get active directions for this sample
            active_dirs = torch.where(direction_vectors[i] > 0)[0]

            # Skip if no directions are active
            if len(active_dirs) == 0:
                continue

            # Apply each active direction
            for dir_idx in active_dirs:
                # Get direction strength
                dir_strength = direction_vectors[i, dir_idx]

                # Apply channel shift with spatial mask
                dir_channel_shift = self.A[dir_idx] * self.scales[dir_idx]
                dir_channel_shift = dir_channel_shift.view(-1, 1, 1)  # (C, 1, 1)

                # Get spatial mask for this direction
                spatial_mask = self.spatial_masks[dir_idx]  # (1, H, W)

                # Calculate contribution from this direction
                dir_delta = dir_channel_shift * spatial_mask * dir_strength * magnitude

                # Add to total delta
                delta_h[i] += dir_delta

        # Return modified bottleneck
        return bottleneck + delta_h

    def get_delta_h(self, direction_idx, magnitude=1.0):
        """
        Get the delta_h for a specific direction (for visualization or analysis)
        """
        channel_shift = self.A[direction_idx] * self.scales[direction_idx]
        channel_shift = channel_shift.unsqueeze(-1).unsqueeze(-1)  # (C, 1, 1)
        spatial_mask = self.spatial_masks[direction_idx]  # (1, H, W)
        return channel_shift * spatial_mask * magnitude

class DiffusionModel:
    def __init__(self, sample_dim, model_name, isPretrained=True,
                 scheduler_cls=DDIMScheduler, num_inference_steps=1000,
                 num_directions=20):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sample_dim = sample_dim    # e.g., (1, 3, 64, 64) or (1, 3, 256, 256)
        self.model_name = model_name
        self.isPretrained = isPretrained
        self.scheduler_cls = scheduler_cls
        self.num_inference_steps = num_inference_steps  # Number of steps for the scheduler
        self.bottleneck = None  # will hold the mid block output during forward pass
        self.num_directions = num_directions

        # Load the main diffusion model
        self.load_model()

        # Initialize auxiliary components
        self._init_auxiliary_components()

    def _init_auxiliary_components(self):
        """Initialize the auxiliary network and regressor."""
        # Get bottleneck dimensions by running a dummy forward pass
        with torch.no_grad():
            dummy_input = torch.randn(1, *self.sample_dim[1:]).to(self.device)
            t = torch.tensor([0]).to(self.device)
            _ = self.model(dummy_input, t)  # This will trigger the forward hook
            bottleneck_shape = self.bottleneck.shape[1:]  # (C, H, W)

        print(f"Detected bottleneck shape: {bottleneck_shape}")

        # Initialize auxiliary network for h-space direction discovery
        self.auxiliary_net = AuxiliaryNetwork(
            bottleneck_channels=bottleneck_shape[0],
            num_directions=self.num_directions,
            bottleneck_spatial_dim=(bottleneck_shape[1], bottleneck_shape[2])
        ).to(self.device)

        # Initialize regressor for direction reconstruction/identification
        self.regressor = DirectionRegressor(
            bottleneck_dim=bottleneck_shape,
            num_directions=self.num_directions
        ).to(self.device)

        # Optimizers
        self.aux_optimizer = optim.Adam(self.auxiliary_net.parameters(), lr=1e-4)
        self.reg_optimizer = optim.Adam(self.regressor.parameters(), lr=1e-4)

        # Loss functions
        self.direction_loss = nn.CrossEntropyLoss()

    def load_model(self):
        if self.isPretrained:
            try:
                # Try to load the pretrained UNet model and move it to the proper device
                self.model = UNet2DModel.from_pretrained(self.model_name).to(self.device)
            except Exception as e:
                print(f"Error loading pretrained model: {e}")
                print("Using a randomly initialized model instead for demonstration")
                # Create a random UNet model if pretrained loading fails
                self.model = UNet2DModel(
                    sample_size=self.sample_dim[-1],  # 256 for 256x256 images
                    in_channels=self.sample_dim[1],   # typically 3 for RGB
                    out_channels=self.sample_dim[1],  # typically 3 for RGB
                    layers_per_block=2,
                    block_out_channels=(128, 256, 512, 512),
                    down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
                    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
                ).to(self.device)

            try:
                # Load the scheduler
                self.scheduler = self.scheduler_cls.from_pretrained(self.model_name, rescale_zero_terminal_snr=True)
            except Exception as e:
                print(f"Error loading scheduler: {e}")
                print("Creating a new scheduler instead")
                # Create a new scheduler if loading fails
                self.scheduler = self.scheduler_cls(
                    beta_start=0.00085,
                    beta_end=0.012,
                    beta_schedule="scaled_linear",
                    num_train_timesteps=1000
                )

            # Set the number of inference steps to initialize the scheduler's timesteps
            self.scheduler.set_timesteps(self.num_inference_steps)

            # Move scheduler internal tensors to the device (if necessary)
            if hasattr(self.scheduler, 'alphas_cumprod'):
                self.scheduler.alphas_cumprod = self.scheduler.alphas_cumprod.to(self.device)

            # Register a forward hook on the mid block to capture its output (bottleneck)
            self.model.mid_block.register_forward_hook(self._save_bottleneck)
        else:
            print("Non-pretrained model loading is not implemented yet.")
            exit()

    def _save_bottleneck(self, module, input, output):
        """
        Hook to capture the output (feature map) of the mid block.
        """
        self.bottleneck = output
        return output  # return the original output without modification

    def inference_step(self, x, t, direction_idx=None, magnitude=1.0, random_directions=False, binary_vectors=None):
        """
        Runs a single inference step through the UNet.
        Returns the noise prediction and the captured bottleneck feature map.

        Args:
            x (torch.Tensor): Input tensor (e.g., a noisy sample)
            t (int or torch.Tensor): The timestep
            direction_idx (int or None): The direction index to apply
            magnitude (float): The strength of the direction modification
            random_directions (bool): Whether to apply random directions
            binary_vectors (tensor or None): Pre-generated binary vectors
        Returns:
            noise_pred (torch.Tensor): The noise prediction from the UNet
            bottleneck (torch.Tensor): The mid block's feature map for this step
            modified_bottleneck (torch.Tensor): The modified bottleneck if applied
        """
        # Ensure timestep is a tensor on the correct device
        if not isinstance(t, torch.Tensor):
            t_tensor = torch.tensor([t]).to(self.device)
        else:
            t_tensor = t.to(self.device)

        # Forward pass (the hook on mid_block will capture the bottleneck)
        with torch.no_grad():
            out = self.model(x, t_tensor)
        noise_pred = out.sample  # assuming the model returns a UNet2DOutput with attribute "sample"

        # Save unmodified bottleneck
        original_bottleneck = self.bottleneck.clone()

        # Optionally modify the bottleneck using the auxiliary network
        modified_bottleneck = None
        if direction_idx is not None or random_directions or binary_vectors is not None:
            modified_bottleneck = self.auxiliary_net(
                original_bottleneck,
                direction_idx=direction_idx,
                magnitude=magnitude,
                random_directions=random_directions,
                binary_vectors=binary_vectors
            )

            # Re-run the remaining layers of the UNet with the modified bottleneck
            # This is a simplified approach; a complete implementation would require
            # more detailed access to the UNet internals
            # For now, we'll use the original noise prediction

        return noise_pred, original_bottleneck, modified_bottleneck

    def diffusion_step(self, x, t, direction_idx=None, magnitude=1.0, random_directions=False, binary_vectors=None):
        """
        Performs one reverse diffusion step: runs inference and then uses the scheduler
        to compute the next (denoised) sample.

        Args:
            x (torch.Tensor): Current sample (noisy image)
            t (int): Current timestep
            direction_idx, magnitude, random_directions, binary_vectors: See inference_step
        Returns:
            new_x (torch.Tensor): The updated sample (x_{t-1})
            bottleneck (torch.Tensor): The bottleneck feature map from the inference step
            modified_bottleneck (torch.Tensor): The modified bottleneck if applied
        """
        # Get noise prediction and bottleneck features
        noise_pred, bottleneck, modified_bottleneck = self.inference_step(
            x, t, direction_idx, magnitude, random_directions, binary_vectors
        )

        # Create a timestep tensor for the scheduler on the correct device
        t_tensor = torch.tensor([t]).to(self.device)
        # Compute the previous sample using the scheduler
        new_x = self.scheduler.step(noise_pred, t_tensor, x).prev_sample
        return new_x, bottleneck, modified_bottleneck

    def train_unsupervised_directions(self, num_iterations=10000, batch_size=16, save_interval=500):
        """
        Train the auxiliary network and regressor to discover interpretable directions
        in an unsupervised manner.
        """
        print(f"Starting unsupervised direction discovery training ({num_iterations} iterations)...")

        # Track losses for monitoring
        losses = {"direction": [], "total": []}

        for iter_idx in tqdm(range(num_iterations)):
            # Generate random noise samples
            x = torch.randn(batch_size, *self.sample_dim[1:]).to(self.device)

            # Select random timestep between 20% and 80% of diffusion process
            t_idx = torch.randint(
                int(0.2 * len(self.scheduler.timesteps)),
                int(0.8 * len(self.scheduler.timesteps)),
                (1,)
            ).item()
            t = self.scheduler.timesteps[t_idx]

            # Generate random binary direction vectors (1-hot or few-hot)
            # For training, we use one-hot encoding to make the task clearer for the regressor
            binary_vectors = torch.zeros(batch_size, self.num_directions, device=self.device)
            direction_indices = []

            for i in range(batch_size):
                # Use one-hot encoding for training
                direction_idx = torch.randint(0, self.num_directions, (1,)).item()
                binary_vectors[i, direction_idx] = 1.0
                direction_indices.append(direction_idx)

            direction_indices = torch.tensor(direction_indices, device=self.device)

            # Run diffusion step with our auxiliary network modifying the bottleneck
            _, _, modified_bottleneck = self.inference_step(
                x, t, binary_vectors=binary_vectors, magnitude=torch.rand(1).item() * 2.0
            )

            # Train the regressor to predict which direction was applied
            self.reg_optimizer.zero_grad()
            direction_preds = self.regressor(modified_bottleneck)
            direction_loss = self.direction_loss(direction_preds, direction_indices)

            # Regularization to encourage diversity in A matrix
            A_diversity_loss = -torch.mean(torch.pdist(self.auxiliary_net.A))

            # Compute total loss
            total_loss = direction_loss + 0.01 * A_diversity_loss

            # Backpropagate and update weights
            total_loss.backward()
            self.reg_optimizer.step()
            self.aux_optimizer.step()

            # Record losses
            losses["direction"].append(direction_loss.item())
            losses["total"].append(total_loss.item())

            # Occasionally report progress
            if iter_idx % 100 == 0:
                print(f"Iteration {iter_idx}: direction loss = {direction_loss.item():.4f}, "
                      f"total loss = {total_loss.item():.4f}")

            # Save checkpoints
            if iter_idx % save_interval == 0:
                torch.save({
                    'auxiliary_net': self.auxiliary_net.state_dict(),
                    'regressor': self.regressor.state_dict(),
                    'iteration': iter_idx,
                    'losses': losses
                }, f"direction_discovery_checkpoint_{iter_idx}.pt")

        print("Unsupervised direction discovery training completed!")
        return losses

    def generate_with_directions(self, batch_size=4, direction_idx=None, magnitude=1.0):
        """
        Generate images with specified direction modifications.

        Args:
            batch_size: Number of images to generate
            direction_idx: Direction index to apply (None for no modification)
            magnitude: Strength of the direction modification

        Returns:
            generated_images: The final generated images
        """
        # Generate initial Gaussian noise
        x = torch.randn(batch_size, *self.sample_dim[1:]).to(self.device)

        # Apply reverse diffusion process
        for t in tqdm(self.scheduler.timesteps, desc="Generating with directions"):
            # Only apply direction modification in the middle of the diffusion process
            # for more stable results
            apply_direction = direction_idx is not None and 0.2 <= t / self.scheduler.timesteps[0] <= 0.8
            dir_idx = direction_idx if apply_direction else None
            mag = magnitude if apply_direction else 0.0

            x, _, _ = self.diffusion_step(x, t, direction_idx=dir_idx, magnitude=mag)

        # Normalize to [0,1] range for visualization
        images = (x * 0.5 + 0.5).clamp(0, 1)
        return images

# Demo function to visualize the learned directions
def visualize_directions(diffusion_model, num_directions=None, magnitudes=[-2.0, -1.0, 0.0, 1.0, 2.0]):
    """
    Generate a grid of images showing the effect of each learned direction.

    Args:
        diffusion_model: The trained diffusion model with auxiliary network
        num_directions: Number of directions to visualize (None = all)
        magnitudes: List of magnitudes to apply for each direction
    """
    import matplotlib.pyplot as plt

    if num_directions is None:
        num_directions = diffusion_model.num_directions
    else:
        num_directions = min(num_directions, diffusion_model.num_directions)

    # Create a figure with rows for each direction and columns for each magnitude
    fig, axes = plt.subplots(num_directions, len(magnitudes),
                             figsize=(len(magnitudes) * 3, num_directions * 3))

    # Generate a single base image without modifications
    base_images = diffusion_model.generate_with_directions(batch_size=1)
    base_image = base_images[0].detach().cpu().permute(1, 2, 0).numpy()

    # For each direction
    for i in range(num_directions):
        # For each magnitude
        for j, magnitude in enumerate(magnitudes):
            if magnitude == 0.0:
                # No modification, use base image
                img = base_image
            else:
                # Generate with the specific direction and magnitude
                modified_images = diffusion_model.generate_with_directions(
                    batch_size=1, direction_idx=i, magnitude=magnitude
                )
                img = modified_images[0].detach().cpu().permute(1, 2, 0).numpy()

            # Display the image
            ax = axes[i, j] if num_directions > 1 else axes[j]
            ax.imshow(img)
            ax.set_title(f"Dir {i}, Mag {magnitude}")
            ax.axis('off')

    plt.tight_layout()
    plt.savefig("discovered_directions.png")
    plt.show()

# Example usage:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    # Sample dimensions for CelebA-HQ (256x256 RGB images)
    sample_dim = (1, 3, 256, 256)
    # Choose a pretrained diffusion model
    model_name = "google/ddpm-ema-celebahq-256"

    # Create the enhanced diffusion model with auxiliary network and regressor
    diffusion_model = DiffusionModel(
        sample_dim=sample_dim,
        model_name=model_name,
        isPretrained=True,
        num_inference_steps=250,
        num_directions=20  # Number of semantic directions to discover
    )

    # Train to discover interpretable directions
    losses = diffusion_model.train_unsupervised_directions(
        num_iterations=5000,  # In practice, might need 10,000+ iterations
        batch_size=8,
        save_interval=500
    )

    # Visualize the discovered directions
    visualize_directions(diffusion_model, num_directions=10)

    # Generate images using Gaussian and Brownian noise with discovered directions
    # Function to generate a Brownian noise trajectory.
    def generate_brownian_noise(num_steps, shape, device="cuda", dt=1.0):
        # Sample Gaussian increments (std = sqrt(dt))
        increments = torch.randn(num_steps, *shape, device=device) * (dt ** 0.5)
        # Cumulative sum along the time dimension to get Brownian noise
        brownian_noise = torch.cumsum(increments, dim=0)
        return brownian_noise

    # Settings
    batch_size = 4
    sample_dim = (batch_size, 3, 256, 256)  # (B, C, H, W)
    num_steps_brownian = 250  # number of steps for the Brownian trajectory

    device = diffusion_model.device

    # Generate initial Gaussian noise for 4 images
    gaussian_noise = torch.randn(sample_dim).to(device)

    # Generate initial Brownian noise for 4 images:
    brownian_noise_list = []
    for i in range(batch_size):
        traj = generate_brownian_noise(num_steps_brownian, sample_dim[1:], device=device)
        brownian_noise_list.append(traj[-1])
    brownian_noise = torch.stack(brownian_noise_list, dim=0)  # shape: (4, 3, 256, 256)

    # Function to display a grid of images
    def show_images(image_tensor, title):
        # image_tensor: shape (batch_size, 3, H, W) with values in [-1, 1]
        imgs = (image_tensor * 0.5 + 0.5).clamp(0, 1).numpy()
        fig, axes = plt.subplots(2, 2, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            # Permute from (C, H, W) to (H, W, C) for display
            img = imgs[i].transpose(1, 2, 0)
            ax.imshow(img)
            ax.axis("off")
        plt.suptitle(title)
        plt.show()

    # Generate images from Gaussian noise with and without direction modifications
    print("Generating from Gaussian noise...")
    x_gaussian = gaussian_noise.clone()
    for t in tqdm(diffusion_model.scheduler.timesteps):
        # Apply a mix of random directions in the middle of the process
        apply_random = 0.3 <= t / diffusion_model.scheduler.timesteps[0] <= 0.7

        if apply_random:
            # Create random binary vectors with 1-3 active directions
            binary_vectors = torch.zeros(batch_size, diffusion_model.num_directions, device=device)
            for i in range(batch_size):
                num_active = torch.randint(1, 4, (1,)).item()
                active_indices = torch.randperm(diffusion_model.num_directions)[:num_active]
                binary_vectors[i, active_indices] = 1.0

            x_gaussian, _, _ = diffusion_model.diffusion_step(
                x_gaussian, t, binary_vectors=binary_vectors, magnitude=1.0
            )
        else:
            x_gaussian, _, _ = diffusion_model.diffusion_step(x_gaussian, t)

    x_gaussian_cpu = x_gaussian.detach().cpu()

    # Generate images from Brownian noise with and without direction modifications
    print("Generating from Brownian noise...")
    x_brownian = brownian_noise.clone()
    for t in tqdm(diffusion_model.scheduler.timesteps):
        # Apply a specific direction in the middle of the process
        apply_direction = 0.3 <= t / diffusion_model.scheduler.timesteps[0] <= 0.7

        if apply_direction:
            # Use direction 0 for the first half of applicable timesteps
            # and direction 1 for the second half
            if t > diffusion_model.scheduler.timesteps[int(len(diffusion_model.scheduler.timesteps) * 0.5)]:
                dir_idx = 0
            else:
                dir_idx = 1

            x_brownian, _, _ = diffusion_model.diffusion_step(
                x_brownian, t, direction_idx=dir_idx, magnitude=1.5
            )
        else:
            x_brownian, _, _ = diffusion_model.diffusion_step(x_brownian, t)

    x_brownian_cpu = x_brownian.detach().cpu()

    # Display results
    show_images(x_gaussian_cpu, "Generated Images from Gaussian Noise with Random Directions")
    show_images(x_brownian_cpu, "Generated Images from Brownian Noise with Specific Directions")